In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

check file is manually filled in, from the report and the TMT files, linking the tube code (e.g. M_H_20) to the corresponding label (amyloid-Beta Status), AIBL.Id, Collection

In [2]:
check_protemics = pd.read_excel('../data/raw_data/check.xlsx', sheet_name='proteomics')
print("Column included: ", check_protemics.columns)
print("Dimension: ", check_protemics.shape)

check_metabolomics = pd.read_excel('../data/raw_data/check.xlsx', sheet_name='metabolomics')
print("Column included: ", check_metabolomics.columns)
print("Dimension: ", check_metabolomics.shape)

Column included:  Index(['label_name', 'Gender', 'label', 'AIBL.ID', 'Collection'], dtype='object')
Dimension:  (90, 5)
Column included:  Index(['label_name', 'Gender', 'label', 'AIBL.ID', 'Collection',
       'Tube Barcode'],
      dtype='object')
Dimension:  (90, 6)


#### For Proteomics

In [3]:
###### no sex covariant
proteomics_df = pd.read_csv('../data/preprocessing/proteomics/ORCA_IRS_COMBAT_nosexcovar_021024_Yihan.csv',index_col=0)
print(proteomics_df.shape)
## use the protein without imputations
proteomics_df = proteomics_df.transpose()
proteomics_df.reset_index(inplace=True)
proteomics_df.rename(columns={'index': 'AIBL.ID'}, inplace=True)
proteomics_df = proteomics_df.apply(lambda col: pd.to_numeric(col, errors='coerce') if col.name != 'label' else col)

(596, 90)


In [4]:
proteomics_cols = proteomics_df.columns.to_list()
proteomics_cols = [value for value in proteomics_cols if value not in ['AIBL.ID']]
print(proteomics_df.select_dtypes(include=['object', 'category']).columns)
proteomics_df = pd.merge(proteomics_df, check_protemics[['Gender','label','AIBL.ID']], on = 'AIBL.ID', how = 'left')

Index([], dtype='object', name='name')


In [5]:
proteomics_df.to_csv('../data/merged_data/proteomics_with_label.csv')

#### For Metabolomics

In [6]:
metabolomics = pd.read_csv('../data/preprocessing/metabolomics/Metabolomics_impute_Normaliseddata_noWave.csv',index_col=0)
metabolomics.reset_index(inplace=True)
metabolomics.rename(columns={'index': 'Tube Barcode'}, inplace=True)
metabolomics['Tube Barcode'] = metabolomics['Tube Barcode'].str.split('_').str[1]
metabolomics_merge_df = pd.merge(metabolomics, check_metabolomics, on = 'Tube Barcode', how = 'right')
metabolomics_merge_df = metabolomics_merge_df.drop(['Tube Barcode', 'Class', 'label_name','Collection'], axis=1)

In [7]:
metabolomics_cols = metabolomics_merge_df.columns.to_list()
metabolomics_cols = [value for value in metabolomics_cols if value not in ['AIBL.ID', 'Gender','label']]

In [8]:
metabolomics_merge_df.to_csv('../data/merged_data/metabolomics_with_label.csv')

#### Merge omics

In [9]:
# merge_df = pd.merge(metabolomics_merge_df,proteomics_merge_df,how = 'inner',on=['AIBL.ID', 'Collection','label'])
merge_df = pd.merge(metabolomics_merge_df,proteomics_df,how = 'inner',on=['AIBL.ID', 'Gender','label'])
print(merge_df.shape)
print(merge_df.select_dtypes(include=['object', 'category']).columns)

(90, 1944)
Index(['Gender', 'label'], dtype='object')


#### Merge with AIBL

In [10]:
AIBL_raw = pd.read_excel('../data/raw_data/AIBL_2023.xlsx')
AIBL_raw['Demographic.YearMonthOfBirth']=AIBL_raw['Demographic.YearMonthOfBirth'].astype(str) + '01'
AIBL_raw['Demographic.YearMonthOfBirth'] = pd.to_datetime(AIBL_raw['Demographic.YearMonthOfBirth'], format='%Y%m%d')
AIBL_raw['Progress Summary.Date of NP assessment'] = pd.to_datetime(AIBL_raw['Progress Summary.Date of NP assessment'], format='%d/%m/%Y')
AIBL_raw['Age'] = (AIBL_raw['Progress Summary.Date of NP assessment'] - AIBL_raw['Demographic.YearMonthOfBirth']).dt.days // 365

In [11]:
AIBL_need = AIBL_raw.copy()
AIBL_need = AIBL_need.rename(columns = {'AIBL ID':'AIBL.ID'})
merge_df = pd.merge(merge_df, check_protemics[['AIBL.ID', 'Collection']], on = ['AIBL.ID'])
omics_merge_aibl_df = pd.merge(merge_df,AIBL_need,how = 'left',on = ['AIBL.ID', 'Collection'])
print("Dimension: ", omics_merge_aibl_df.shape)

Dimension:  (90, 3344)


In [12]:
omics_merge_aibl_df.to_csv('../data/merged_data/omics_merge_aibl.csv')

omics with demographic information

In [13]:
print(merge_df.shape)

aibl_demo_df = omics_merge_aibl_df[['AIBL.ID', 'Collection', 'Demographic.ApoE genotype','Demographic.Years of Education','Demographic.Years of Education Exact', 'Age', 'Demographic.Sex',
                                    'Demographic.Primary Occupation', 'Demographic.Retirement Status','Demographic.Current Marital Status', 'Demographic.Country of Birth', 'Demographic.Handedness', 'Vital Signs.Body Mass Index', 'Demographic.GDS']]
aibl_demo_df['APOE4'] = np.where(
aibl_demo_df['Demographic.ApoE genotype'].str.contains('E4', na=False),
    1,
    np.where(aibl_demo_df['Demographic.ApoE genotype'].isna(),np.nan,0)
)
omics_with_demo_df = pd.merge(merge_df, aibl_demo_df, on = ['AIBL.ID', 'Collection'], how='inner')
print(omics_with_demo_df.shape)

(90, 1945)
(90, 1958)


In [14]:
omics_with_demo_df.to_csv('../data/merged_data/omics_aibl_demo.csv',index=False)

In [15]:
missing_values = omics_with_demo_df.isnull().sum()
missing_info = {col: count for col, count in missing_values.items() if count > 0}
print(missing_info)

{'Demographic.ApoE genotype': 2, 'Demographic.Years of Education Exact': 6, 'Demographic.Primary Occupation': 2, 'Demographic.Retirement Status': 1, 'Demographic.Current Marital Status': 1, 'Demographic.Country of Birth': 1, 'Demographic.Handedness': 1, 'Vital Signs.Body Mass Index': 10, 'APOE4': 2}


In [ ]:
######## Go to R file (imputation.Rmd) to impute only demographic information

In [16]:
combined_features = [(name, 'proteomics') for name in proteomics_cols] + \
                    [(name, 'metabolomics') for name in metabolomics_cols] + \
                    [(name, 'demographic') for name in ['Demographic.Sex','APOE4','Age','Demographic.Years of Education Exact']]

features_df = pd.DataFrame(combined_features, columns=['feature', 'type'])

features_df.to_csv('../data/merged_data/feature_type.csv')